In [21]:
#!/usr/bin/env python3
"""
NHL team stats scraper for https://www.scrapethissite.com/pages/forms/

What it does:
1) Scrapes ALL paginated pages (since 1990)
2) Writes the full dataset to data.csv
3) Loads data.csv with pandas
4) Answers:
   - Who made the most wins in 1990, 2000, 2010?
   - How many teams participated in 1991, 2001, 2011?

Run:
  python nhl_scrape.py

(If you’re using Jupyter, just paste the whole file into one notebook cell or split it into cells.)
"""

import time
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://www.scrapethissite.com/pages/forms/"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

# This fallback matches the site’s hockey table structure (9 columns).
FALLBACK_COLUMNS = [
    "Team Name",
    "Year",
    "Wins",
    "Losses",
    "OT Losses",
    "Win %",
    "Goals For (GF)",
    "Goals Against (GA)",
    "+ / -",
]


def fetch_page(page_num: int, per_page: int = 100) -> BeautifulSoup:
    params = {"page_num": page_num, "per_page": per_page}
    r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def parse_columns(soup: BeautifulSoup) -> list[str]:
    """
    Try to read the visible table headers. If not found (some HTML variations),
    return a known-good fallback list of column names.
    """
    # More robust than "thead th": sometimes the page markup differs
    ths = soup.select("table.table th")
    cols = [th.get_text(strip=True) for th in ths if th.get_text(strip=True)]

    # We expect exactly 9 columns for this dataset
    if len(cols) == 9:
        return cols

    return FALLBACK_COLUMNS.copy()


def parse_rows(soup: BeautifulSoup) -> list[list[str]]:
    """
    Parse rows by cell class names, which is robust when OT Losses is missing.
    """
    rows: list[list[str]] = []

    for tr in soup.select("tr.team"):
        def txt(selector: str, default: str = "") -> str:
            el = tr.select_one(selector)
            return el.get_text(strip=True) if el else default

        rows.append([
            txt("td.name"),
            txt("td.year"),
            txt("td.wins"),
            txt("td.losses"),
            txt("td.ot-losses", "0"),  # missing on some older seasons
            txt("td.pct"),
            txt("td.gf"),
            txt("td.ga"),
            txt("td.diff"),
        ])

    return rows


def scrape_all(per_page: int = 100, sleep_s: float = 0.2) -> pd.DataFrame:
    all_rows: list[list[str]] = []
    columns: list[str] | None = None

    page_num = 1
    while True:
        soup = fetch_page(page_num=page_num, per_page=per_page)

        if columns is None:
            columns = parse_columns(soup)
            print("Columns:", columns)

        page_rows = parse_rows(soup)
        print(f"Page {page_num}: {len(page_rows)} rows")

        if not page_rows:
            break

        all_rows.extend(page_rows)
        page_num += 1
        time.sleep(sleep_s)

    if columns is None or len(columns) == 0:
        columns = FALLBACK_COLUMNS.copy()

    df = pd.DataFrame(all_rows, columns=columns)
    return df


def clean_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert columns to numeric types where appropriate.
    This makes the pandas analysis reliable.
    """
    # Rename to fallback names if the site header came back weird but data matches 9 fields
    # (Keeps headers stable for later analysis.)
    if list(df.columns) != FALLBACK_COLUMNS and len(df.columns) == 9:
        # Keep original headers for the CSV (assignment wants HTML headers),
        # but analysis is easier if we can locate known columns.
        # We'll create "analysis columns" safely below.
        pass

    # Build a mapping from "whatever the header is" -> expected meaning, if possible.
    # If the scraped headers equal the fallback, mapping is identity.
    def find_col(possible_names: list[str]) -> str | None:
        for name in possible_names:
            if name in df.columns:
                return name
        return None

    col_year = find_col(["Year"])
    col_team = find_col(["Team Name"])
    col_wins = find_col(["Wins"])

    # Some versions might name Win% slightly differently; not essential for questions
    # Convert core numeric columns if they exist
    for c in ["Year", "Wins", "Losses", "OT Losses", "Goals For (GF)", "Goals Against (GA)", "+ / -"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Ensure required columns exist for analysis
    if col_year is None or col_team is None or col_wins is None:
        # If headers differ, try position-based fallback since our row parser is fixed order
        # Team, Year, Wins, Losses, OT, Pct, GF, GA, Diff
        df.columns = FALLBACK_COLUMNS
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
        df["Wins"] = pd.to_numeric(df["Wins"], errors="coerce")

    return df


def answer_questions(df: pd.DataFrame) -> None:
    # Q1: most wins in 1990, 2000, 2010
    years_q1 = [1990, 2000, 2010]
    most_wins = (
        df[df["Year"].isin(years_q1)]
        .sort_values(["Year", "Wins"], ascending=[True, False])
        .groupby("Year", as_index=False)
        .head(1)[["Year", "Team Name", "Wins"]]
        .sort_values("Year")
    )

    print("\nQ1) Most wins:")
    for _, row in most_wins.iterrows():
        print(f"  {int(row['Year'])}: {row['Team Name']} ({int(row['Wins'])} wins)")

    # Q2: number of teams in 1991, 2001, 2011
    years_q2 = [1991, 2001, 2011]
    teams_count = (
        df[df["Year"].isin(years_q2)]
        .groupby("Year")["Team Name"]
        .nunique()
        .reset_index(name="Number of teams")
        .sort_values("Year")
    )

    print("\nQ2) Teams participated:")
    for _, row in teams_count.iterrows():
        print(f"  {int(row['Year'])}: {int(row['Number of teams'])} teams")


def main():
    df_scraped = scrape_all(per_page=100, sleep_s=0.2)

    # Write CSV with whatever headers we scraped (assignment requirement)
    df_scraped.to_csv("data.csv", index=False)
    print("\nSaved:", os.path.abspath("data.csv"))
    print("Rows saved:", len(df_scraped))

    # Load and analyze with pandas
    df = pd.read_csv("data.csv")
    df = clean_types(df)
    answer_questions(df)


if __name__ == "__main__":
    main()


Columns: ['Team Name', 'Year', 'Wins', 'Losses', 'OT Losses', 'Win %', 'Goals For (GF)', 'Goals Against (GA)', '+ / -']
Page 1: 100 rows
Page 2: 100 rows
Page 3: 100 rows
Page 4: 100 rows
Page 5: 100 rows
Page 6: 82 rows
Page 7: 0 rows

Saved: /Users/joannkleinheyer/DIS08/data.csv
Rows saved: 582

Q1) Most wins:
  1990: Chicago Blackhawks (49 wins)
  2000: Colorado Avalanche (52 wins)
  2010: Vancouver Canucks (54 wins)

Q2) Teams participated:
  1991: 22 teams
  2001: 30 teams
  2011: 30 teams
